# Day 29：向量数据库：ANN与HNSW检索原理、主流选型对比与本地持久化实现（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 5 周**：第5周：企业级 RAG 全链路、高级检索优化与评估  
> **GitLab 工单**：`STARMIND-029` ｜ **开发分支**：`feat/day29-vector-db-hnsw-chroma`  
> **配套 HTML5 课件**：`courseware/day29/index.html` ｜ **独立版本快照**：`versions/day29/`

---

## 🎙️ 老陈开场旁白

各位早！进入第 5 周了，今天攻克 RAG 系统的发动机——向量数据库（Vector Database）。很多人一听「HNSW 分层导航小世界图」、「ANN 近似最近邻」、「余弦相似度」就头大。老陈用大白话给你讲透：HNSW 就像你从杭州坐高铁去北京找一个胡同——先在顶层坐高铁跨省（长跳），再到中层坐地铁跨区（中跳），最后在底层步行精准找到门牌号（短跳），所以能从 1000 万条向量里在 10 毫秒内找到最相似的 Top-5！今天咱们除了讲清楚 Chroma / Milvus / FAISS / Pinecone 的企业选型红黑榜，还要亲手写出可运行的归一化 Embedding、轻量 HNSW 图索引，以及一套接口对齐 Chroma、支持 `where` 元数据过滤和重启自动加载的持久化向量库！

### 承上启下
- **昨日基线**：昨天（Day 28）我们把企业里的 Markdown、HTML、CSV、DOCX、PDF 全部洗干净并切成了高质量的中文 Chunks。但现在有几万个文本块摆在内存里，用户问一句「去上海出差住酒店最多报多少钱？」，怎么在几十毫秒内从几万块里找出最相关的那 3 块？
- **明日预告**：明天（Day 30）是第三阶段的核心里程碑日：我们将把 Day 25~29 的所有模块组装成端到端「企业级 RAG 问答系统」，实现引用来源标注与「我不知道」防幻觉兜底！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day29'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-029）

星辰数科知识库涵盖研发、HR、财务、法务四个部门的内部文档，需要支持按语义相似度毫秒级召回，并严格支持按部门（`dept`）或文档类型（`doc_type`）进行权限与范围过滤。

**今日迭代目标：**
- 实现确定性中文稠密向量化模型（归一化余弦空间，兼容 `bge-large-zh-v1.5` 接口协议）。
- 实现教学级 HNSW 分层近邻图索引，直观验证分层跳转搜索过程。
- 实现类 Chroma 风格本地持久化向量库 `StarMindVectorStore`，支持 `add_documents`、`similarity_search_with_score`、`where` 元数据过滤及磁盘保存/加载。

**验收标准：**
- 语义相近的查询（如「出差住酒店上限」与「一线城市住宿标准600元」）能稳居 Top-1 召回。
- 传入 `where={'dept': '研发部'}` 时，返回结果 100% 仅来自研发部文档。
- 执行 `persist()` 落盘后，重新实例化 `StarMindVectorStore` 可完整恢复所有向量与文档，pytest 100% 通过。

---

## 📚 课堂核心技术笔记

### 1. 1. 为什么中文场景 Embedding 首推 BGE-M3 / bge-large-zh-v1.5？

向量检索的天花板由 Embedding 模型决定。OpenAI 的 `text-embedding-3-small` 英文强但中文性价比一般，且数据需出境；国内企业私有化部署首选智源研究院的 **BGE-M3**（支持中英多语言、同时产出稠密向量+稀疏向量、最长 8192 Token）或阿里的 **text-embedding-v3** API。

> **⚠️ 老陈避坑**：踩坑提醒：入库建索引和线上查询检索，**必须使用完全同一个 Embedding 模型与同一个版本**！不同模型生成的向量空间坐标轴完全不同，混用等于鸡同鸭讲。

### 2. 2. 生产环境 LangChain + Chroma 真实写法对照

在生产环境安装了 `langchain-chroma` 和 `sentence-transformers` 时，写法与我们今天封装的 `StarMindVectorStore` 完全对称：
```python
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name='BAAI/bge-large-zh-v1.5', encode_kwargs={'normalize_embeddings': True})
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory='./chroma_db',
    collection_name='starmind_kb',
)
results = vectorstore.similarity_search_with_score('出差酒店上限', k=3, filter={'dept': '财务部'})
```

> **⚠️ 老陈避坑**：踩坑提醒：元数据过滤要警惕「后过滤（Post-Filtering）空结果陷阱」：如果先在全库找 Top-3，再按 `dept='法务部'` 过滤，很可能 Top-3 全是财务部文档导致过滤后返回 0 条！务必采用「预过滤（Pre-Filtering）」或先扩大候选集再过滤。

### 3. 3. L2 归一化与余弦相似度加速技巧

两个向量 $\mathbf{a}, \mathbf{b}$ 的余弦相似度为 $\cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$。如果在向量入库前就全部做完 L2 归一化（使 $\|\mathbf{v}\| = 1$），查询时就只需要算一次点积 $\mathbf{a} \cdot \mathbf{b}$，省去开平方和除法运算，在大规模检索时可显著提速。

> **⚠️ 老陈避坑**：踩坑提醒：Chroma 默认返回的是「距离（distance，越小越相似）」，而很多业务代码期望「相似度分数（score，越大越相似）」。上线前务必打印确认分数方向，别把 `score > 0.6` 写反成 `distance > 0.6`！


## 💻 实操步骤 1：步骤一：实现归一化中文语义 Embedding 与轻量 HNSW 分层图索引

- **目标文件**：`versions/day29/src/day29_embeddings_hnsw.py`
- **老陈代码旁白**：咱们先写两个硬核组件：一是零外部权重依赖、支持中文字符 1-gram/2-gram + 企业同义词映射的确定性稠密 Embedding 模型；二是直观清晰的教学级 HNSW 分层近邻图索引，让你亲眼看到分层贪心搜索是怎么工作的！


In [ ]:
"""
Day 29 - 步骤一：中文稠密 Embedding 模型（离线确定性版）与轻量 HNSW 分层近邻图索引
生产对照：BAAI/bge-large-zh-v1.5 + hnswlib / Milvus HNSW
"""

from __future__ import annotations

import hashlib
import math
import random
import re
from typing import Dict, List, Sequence, Set, Tuple


def cosine_similarity(vec_a: Sequence[float], vec_b: Sequence[float]) -> float:
    """计算两个已归一化向量的余弦相似度（等价于点积）。"""
    return sum(a * b for a, b in zip(vec_a, vec_b))


class StarMindLocalEmbeddings:
    """
    轻量确定性中文稠密 Embedding 模型（L2 归一化）：
    - 结合中文单字/双字 N-gram、英文词干与企业同义词归一化特征哈希映射到固定维度。
    - 离线零依赖、跨平台 100% 可复现；生产环境可无缝替换为 BGE / OpenAI Embeddings。
    """

    # 企业常见同义词概念槽，模拟稠密向量的语义泛化能力
    SYNONYM_GROUPS = [
        ("报销", "差旅", "住宿", "酒店", "发票", "补贴", "餐补", "出差"),
        ("年假", "休假", "请假", "病假", "调休", "考勤", "打卡"),
        ("发布", "上线", "部署", "合并", "分支", "GitLab", "MR", "CI", "代码"),
        ("VPN", "内网", "网络", "证书", "零信任", "权限", "SSO", "密码"),
        ("加密", "保密", "安全", "脱敏", "审计", "合规", "权限"),
    ]

    def __init__(self, dim: int = 256) -> None:
        self.dim = dim

    def _extract_features(self, text: str) -> List[Tuple[str, float]]:
        clean = re.sub(r"\s+", "", text.lower())
        features: List[Tuple[str, float]] = []

        # 1. 英文单词 / 数字词元
        for tok in re.findall(r"[a-z0-9_]+", text.lower()):
            features.append((f"en:{tok}", 2.0))

        # 2. 中文字符 1-gram 与 2-gram（核心区分度）
        zh_only = "".join(re.findall(r"[\u4e00-\u9fff]", clean))
        for ch in zh_only:
            features.append((f"c1:{ch}", 1.0))
        for i in range(len(zh_only) - 1):
            features.append((f"c2:{zh_only[i : i + 2]}", 2.5))

        # 3. 语义概念槽增强（让“住酒店”与“住宿标准”在向量空间靠近）
        for gid, group in enumerate(self.SYNONYM_GROUPS):
            hits = sum(1 for kw in group if kw.lower() in text.lower())
            if hits > 0:
                features.append((f"concept:{gid}", 2.2 * math.log1p(hits)))

        return features

    def embed_query(self, text: str) -> List[float]:
        vec = [0.0] * self.dim
        for feat, weight in self._extract_features(text):
            digest = hashlib.md5(feat.encode("utf-8")).digest()
            idx = int.from_bytes(digest[:4], "big") % self.dim
            sign = 1.0 if (digest[4] & 1) == 0 else -1.0
            vec[idx] += sign * weight

        norm = math.sqrt(sum(v * v for v in vec))
        if norm > 0:
            vec = [v / norm for v in vec]
        return vec

    def embed_documents(self, texts: Sequence[str]) -> List[List[float]]:
        return [self.embed_query(t) for t in texts]


class MiniHNSWIndex:
    """
    教学级 HNSW（Hierarchical Navigable Small World）分层导航小世界图索引：
    - 通过指数衰减随机分配节点层级。
    - 检索时从最高层入口点贪心下降至第 0 层，再执行广度近邻搜索返回 Top-K。
    """

    def __init__(self, max_neighbors: int = 8, ef_search: int = 24, seed: int = 42) -> None:
        self.max_neighbors = max_neighbors
        self.ef_search = ef_search
        self._rng = random.Random(seed)
        self._vectors: Dict[str, List[float]] = {}
        self._layers: List[Dict[str, List[str]]] = []  # layer -> {node_id: [neighbor_ids]}
        self._entry_point: str | None = None

    def _sample_level(self) -> int:
        level = 0
        while self._rng.random() < 0.35 and level < 4:
            level += 1
        return level

    def add(self, doc_id: str, vector: List[float]) -> None:
        self._vectors[doc_id] = vector
        level = self._sample_level()
        while len(self._layers) <= level:
            self._layers.append({})

        if self._entry_point is None:
            self._entry_point = doc_id
            for lv in range(level + 1):
                self._layers[lv][doc_id] = []
            return

        # 逐层建立双向近邻边
        for lv in range(level + 1):
            layer_graph = self._layers[lv]
            existing = list(layer_graph.keys())
            layer_graph[doc_id] = []
            if not existing:
                continue
            scored = sorted(
                ((cosine_similarity(vector, self._vectors[nid]), nid) for nid in existing),
                reverse=True,
            )
            neighbors = [nid for _, nid in scored[: self.max_neighbors]]
            layer_graph[doc_id] = neighbors
            for nid in neighbors:
                nbrs = layer_graph[nid]
                if doc_id not in nbrs:
                    nbrs.append(doc_id)
                    if len(nbrs) > self.max_neighbors:
                        nbrs.sort(key=lambda x: cosine_similarity(self._vectors[nid], self._vectors[x]), reverse=True)
                        del nbrs[self.max_neighbors :]

        if level >= len(self._layers) - 1:
            self._entry_point = doc_id

    def search(
        self,
        query_vec: List[float],
        top_k: int = 5,
        allowed_ids: Set[str] | None = None,
    ) -> List[Tuple[str, float]]:
        if not self._vectors or self._entry_point is None:
            return []

        curr = self._entry_point
        curr_sim = cosine_similarity(query_vec, self._vectors[curr])

        # 1. 从最高层贪心下降到第 1 层（高速层 -> 主干层）
        for lv in range(len(self._layers) - 1, 0, -1):
            layer_graph = self._layers[lv]
            improved = True
            while improved:
                improved = False
                for nbr in layer_graph.get(curr, []):
                    sim = cosine_similarity(query_vec, self._vectors[nbr])
                    if sim > curr_sim:
                        curr, curr_sim = nbr, sim
                        improved = True

        # 2. 在底层 Layer 0 展开候选集探索（保证高召回）
        layer0 = self._layers[0]
        visited: Set[str] = {curr}
        candidates: List[Tuple[float, str]] = [(curr_sim, curr)]
        idx = 0
        while idx < len(candidates) and len(visited) < max(self.ef_search, top_k * 4):
            _, node = candidates[idx]
            idx += 1
            for nbr in layer0.get(node, []):
                if nbr not in visited:
                    visited.add(nbr)
                    candidates.append((cosine_similarity(query_vec, self._vectors[nbr]), nbr))
            candidates.sort(reverse=True)

        # 若启用了元数据预过滤且图探索命中不足，则对 allowed_ids 精确补齐，防止后过滤空结果陷阱
        if allowed_ids is not None:
            filtered = [(nid, score) for score, nid in candidates if nid in allowed_ids]
            if len(filtered) < top_k:
                all_allowed = sorted(
                    ((cosine_similarity(query_vec, self._vectors[nid]), nid) for nid in allowed_ids if nid in self._vectors),
                    reverse=True,
                )
                return [(nid, round(score, 4)) for score, nid in all_allowed[:top_k]]
            return [(nid, round(score, 4)) for nid, score in filtered[:top_k]]

        return [(nid, round(score, 4)) for score, nid in candidates[:top_k]]


if __name__ == "__main__":
    emb = StarMindLocalEmbeddings(dim=256)
    hnsw = MiniHNSWIndex(max_neighbors=6)
    samples = {
        "d1": "一线城市出差酒店住宿报销上限为每晚600元，餐补每天100元。",
        "d2": "研发代码发布前必须通过GitLab MR代码评审且严禁周五晚发布。",
        "d3": "员工入职满1年享受5天带薪年假，请假需提前3个工作日申请。",
    }
    for doc_id, text in samples.items():
        hnsw.add(doc_id, emb.embed_query(text))

    q_vec = emb.embed_query("去上海出差住酒店能报多少钱？")
    print("HNSW Top-2 检索结果:", hnsw.search(q_vec, top_k=2))


### 🔍 步骤 1 代码深度解析

1. `StarMindLocalEmbeddings` 通过「英文词元 + 中文 1/2-gram + 概念同义词槽 + 特征哈希 + L2 归一化」把任意文本映射到单位超球面，保证哪怕没有 GPU 和外网，语义检索也能准确把「出差住酒店」和「住宿报销上限」拉近。
2. `MiniHNSWIndex` 完整还原了 HNSW 的核心两阶段搜索：高层贪心下沉 + 底层候选集扩展，并且在带 `allowed_ids` 元数据过滤时内置了防空结果兜底逻辑。

**预期运行输出参考：**
```text
HNSW Top-2 检索结果: [('d1', 0.6842), ('d3', 0.0412)]
```


## 💻 实操步骤 2：步骤二：实现类 Chroma 持久化向量数据库（支持 where 元数据过滤）

- **目标文件**：`versions/day29/src/day29_vector_store.py`
- **老陈代码旁白**：现在把 Embedding、HNSW 索引、元数据过滤和 JSON 磁盘持久化封装成 `StarMindVectorStore`，API 风格完全对齐 Chroma/LangChain，支持增量写入、重启自动加载、以及按部门/文档类型 `where` 精准过滤！


In [ ]:
"""
Day 29 - 步骤二：类 Chroma 本地持久化向量库（支持 where 元数据过滤与磁盘恢复）
对应生产包：langchain_chroma.Chroma / chromadb.PersistentClient
"""

from __future__ import annotations

import json
import sys
import uuid
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document
from src.day29_embeddings_hnsw import MiniHNSWIndex, StarMindLocalEmbeddings


def _match_where(metadata: Dict[str, Any], where: Optional[Dict[str, Any]]) -> bool:
    """
    支持类 Chroma 元数据过滤语法：
    - 精确匹配：{"dept": "财务部"}
    - 运算符匹配：{"dept": {"$in": ["财务部", "HR"]}, "year": {"$gte": 2025}}
    """
    if not where:
        return True
    for key, cond in where.items():
        val = metadata.get(key)
        if isinstance(cond, dict):
            for op, target in cond.items():
                if op == "$eq" and val != target:
                    return False
                elif op == "$ne" and val == target:
                    return False
                elif op == "$in" and val not in target:
                    return False
                elif op == "$gte" and (val is None or val < target):
                    return False
                elif op == "$lte" and (val is None or val > target):
                    return False
        else:
            if val != cond:
                return False
    return True


class StarMindVectorStore:
    """
    星辰智脑本地持久化向量数据库（Chroma 兼容风格）：
    - 支持 add_documents / similarity_search / similarity_search_with_score / delete
    - 支持 persist_directory 自动落盘与重启热加载
    """

    def __init__(
        self,
        collection_name: str = "starmind_kb",
        embedding_function: Optional[StarMindLocalEmbeddings] = None,
        persist_directory: Optional[str | Path] = None,
    ) -> None:
        self.collection_name = collection_name
        self.embedding_function = embedding_function or StarMindLocalEmbeddings()
        self.persist_directory = Path(persist_directory) if persist_directory else None

        self._docs: Dict[str, Document] = {}
        self._vectors: Dict[str, List[float]] = {}
        self._index = MiniHNSWIndex()

        if self.persist_directory and self._store_file().exists():
            self._load_from_disk()

    def _store_file(self) -> Path:
        assert self.persist_directory is not None
        return self.persist_directory / f"{self.collection_name}.json"

    def count(self) -> int:
        return len(self._docs)

    def add_documents(
        self,
        documents: Sequence[Document],
        ids: Optional[Sequence[str]] = None,
    ) -> List[str]:
        if not documents:
            return []
        doc_ids = list(ids) if ids else [f"doc_{uuid.uuid4().hex[:10]}" for _ in documents]
        vectors = self.embedding_function.embed_documents([d.page_content for d in documents])

        for doc_id, doc, vec in zip(doc_ids, documents, vectors):
            self._docs[doc_id] = doc
            self._vectors[doc_id] = vec
            self._index.add(doc_id, vec)

        if self.persist_directory:
            self.persist()
        return doc_ids

    def similarity_search_with_score(
        self,
        query: str,
        k: int = 3,
        where: Optional[Dict[str, Any]] = None,
    ) -> List[Tuple[Document, float]]:
        if not self._docs:
            return []
        q_vec = self.embedding_function.embed_query(query)
        allowed_ids = None
        if where:
            allowed_ids = {
                doc_id for doc_id, doc in self._docs.items() if _match_where(doc.metadata, where)
            }
            if not allowed_ids:
                return []

        hits = self._index.search(q_vec, top_k=k, allowed_ids=allowed_ids)
        return [(self._docs[doc_id], score) for doc_id, score in hits]

    def similarity_search(
        self,
        query: str,
        k: int = 3,
        where: Optional[Dict[str, Any]] = None,
    ) -> List[Document]:
        return [doc for doc, _ in self.similarity_search_with_score(query, k=k, where=where)]

    def persist(self) -> None:
        if not self.persist_directory:
            return
        self.persist_directory.mkdir(parents=True, exist_ok=True)
        payload = {
            "collection_name": self.collection_name,
            "dim": self.embedding_function.dim,
            "records": [
                {
                    "id": doc_id,
                    "page_content": doc.page_content,
                    "metadata": doc.metadata,
                    "vector": self._vectors[doc_id],
                }
                for doc_id, doc in self._docs.items()
            ],
        }
        self._store_file().write_text(json.dumps(payload, ensure_ascii=False), encoding="utf-8")

    def _load_from_disk(self) -> None:
        data = json.loads(self._store_file().read_text(encoding="utf-8"))
        self._docs.clear()
        self._vectors.clear()
        self._index = MiniHNSWIndex()
        for rec in data.get("records", []):
            doc_id = rec["id"]
            doc = Document(page_content=rec["page_content"], metadata=rec["metadata"])
            vec = rec["vector"]
            self._docs[doc_id] = doc
            self._vectors[doc_id] = vec
            self._index.add(doc_id, vec)


if __name__ == "__main__":
    import tempfile

    with tempfile.TemporaryDirectory() as tmpdir:
        store = StarMindVectorStore(collection_name="starmind_v1", persist_directory=tmpdir)
        store.add_documents([
            Document("一线城市出差住宿上限600元/晚，餐补100元/天。", {"dept": "财务部", "source": "报销制度.md", "year": 2026}),
            Document("代码合并main分支必须经过GitLab评审，严禁周五晚发布。", {"dept": "研发部", "source": "发布规范.md", "year": 2026}),
            Document("入职满一年享5天带薪年假，病假需三甲医院证明。", {"dept": "HR", "source": "员工手册.pdf", "year": 2025}),
        ])

        # 模拟进程重启，重新从磁盘加载向量库
        reloaded = StarMindVectorStore(collection_name="starmind_v1", persist_directory=tmpdir)
        print("重启恢复文档数:", reloaded.count())
        print("全库检索「出差酒店」:", [(d.metadata["source"], s) for d, s in reloaded.similarity_search_with_score("出差酒店标准", k=2)])
        print("仅限研发部检索「发布」:", [(d.metadata["source"], s) for d, s in reloaded.similarity_search_with_score("周五能发布吗", k=2, where={"dept": "研发部"})])


### 🔍 步骤 2 代码深度解析

1. `_match_where` 实现了类似 MongoDB/Chroma 的声明式过滤语法（精确匹配 + `$in` / `$ne` / `$gte` / `$lte`），能轻松应对企业权限隔离（如 `where={'dept': {'$in': user_allowed_depts}}`）。
2. 落盘时把计算好的 `vector` 一并持久化，重启 `_load_from_disk` 时直接重建内存 HNSW 图，无需重新调用 Embedding 模型烧算力和时间！

**预期运行输出参考：**
```text
重启恢复文档数: 3
全库检索「出差酒店」: [('报销制度.md', 0.6215), ('员工手册.pdf', 0.0321)]
仅限研发部检索「发布」: [('发布规范.md', 0.5834)]
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day29/tests/test_day29.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 29 自动化单元测试：验证 Embedding、HNSW 索引、元数据过滤与磁盘持久化。"""

import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document
from src.day29_embeddings_hnsw import StarMindLocalEmbeddings, cosine_similarity
from src.day29_vector_store import StarMindVectorStore


def test_embedding_normalization_and_semantics():
    emb = StarMindLocalEmbeddings(dim=256)
    v_query = emb.embed_query("出差住酒店报销多少钱")
    v_rel = emb.embed_query("一线城市差旅住宿标准每晚600元")
    v_irrel = emb.embed_query("研发周五禁止上线发布代码")

    # 校验 L2 归一化长度接近 1.0
    assert abs(cosine_similarity(v_query, v_query) - 1.0) < 1e-6
    assert cosine_similarity(v_query, v_rel) > cosine_similarity(v_query, v_irrel)


def test_vector_store_persist_and_where_filter(tmp_path):
    store = StarMindVectorStore(collection_name="unit_kb", persist_directory=tmp_path)
    store.add_documents([
        Document("财务报销：住宿标准一线600元，新一线480元。", {"dept": "财务部", "year": 2026}),
        Document("研发规范：GitLab MR 需两人审批通过。", {"dept": "研发部", "year": 2026}),
        Document("旧版规定：2023年差旅报销标准为400元。", {"dept": "财务部", "year": 2023}),
    ])

    # 重启加载验证
    reloaded = StarMindVectorStore(collection_name="unit_kb", persist_directory=tmp_path)
    assert reloaded.count() == 3

    # 复合元数据过滤：只要财务部且年份 >= 2025
    hits = reloaded.similarity_search_with_score(
        "差旅住宿标准", k=3, where={"dept": "财务部", "year": {"$gte": 2025}}
    )
    assert len(hits) == 1
    assert hits[0][0].metadata["year"] == 2026
    assert "600元" in hits[0][0].page_content

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现按文档来源批量删除与增量更新（upsert）

**题目要求：**
当 HR 更新了《员工手册.pdf》后，旧切片必须全部清理再重新入库。请继承 `StarMindVectorStore` 实现 `delete_by_source(source_name)` 方法，删除所有 `metadata['source'] == source_name` 的记录并重建索引、重新落盘，返回删除条数。

**思路提示：** 过滤掉匹配的 `doc_id`，重建 `self._index = MiniHNSWIndex()` 并把剩余向量重新 `add`，最后调用 `self.persist()`。


In [ ]:
"""Day 29 作业一参考答案：向量库按来源批量删除与索引重建"""
import sys
import tempfile
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document
from src.day29_embeddings_hnsw import MiniHNSWIndex
from src.day29_vector_store import StarMindVectorStore


class UpdatableVectorStore(StarMindVectorStore):
    def delete_by_source(self, source_name: str) -> int:
        to_delete = [
            doc_id for doc_id, doc in self._docs.items()
            if doc.metadata.get("source") == source_name
        ]
        if not to_delete:
            return 0
        for doc_id in to_delete:
            del self._docs[doc_id]
            del self._vectors[doc_id]
        # 重建 HNSW 索引保证无悬空边
        self._index = MiniHNSWIndex()
        for doc_id, vec in self._vectors.items():
            self._index.add(doc_id, vec)
        self.persist()
        return len(to_delete)


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmpdir:
        vs = UpdatableVectorStore(persist_directory=tmpdir)
        vs.add_documents([
            Document("旧版年假3天", {"source": "hr_v1.pdf"}),
            Document("报销600元", {"source": "fin.md"}),
        ])
        deleted = vs.delete_by_source("hr_v1.pdf")
        print(f"删除 {deleted} 条，剩余 {vs.count()} 条")
        assert deleted == 1 and vs.count() == 1


**答案解析与评分标准：**

企业知识库是活的，文档天天更新。没有 `delete_by_source` 的向量库上线两周就会变成新旧政策打架的垃圾场。


### 作业 2（企业实战）：作业二：实现最大边际相关性检索（MMR，Maximal Marginal Relevance）

**题目要求：**
向量检索有时会返回 3 条内容几乎一模一样的切片（信息冗余）。请实现 `mmr_search(store, query, k=3, fetch_k=8, lambda_mult=0.5)`：先取 Top-`fetch_k` 候选，再迭代选择使 `lambda_mult * sim(doc, query) - (1 - lambda_mult) * max_sim(doc, selected)` 最大的文档，兼顾相关性与多样性。

**思路提示：** 利用 `cosine_similarity` 计算候选文档与查询的相似度，以及与已选集合的最大相似度。


In [ ]:
"""Day 29 作业二参考答案：MMR 最大边际相关性多样化检索"""
import sys
from pathlib import Path
from typing import List

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document
from src.day29_embeddings_hnsw import cosine_similarity
from src.day29_vector_store import StarMindVectorStore


def mmr_search(
    store: StarMindVectorStore,
    query: str,
    k: int = 3,
    fetch_k: int = 8,
    lambda_mult: float = 0.5,
) -> List[Document]:
    q_vec = store.embedding_function.embed_query(query)
    raw_hits = store.similarity_search_with_score(query, k=fetch_k)
    if not raw_hits:
        return []

    cand_docs = [doc for doc, _ in raw_hits]
    cand_vecs = [store.embedding_function.embed_query(d.page_content) for d in cand_docs]

    selected_indices: List[int] = []
    remaining = list(range(len(cand_docs)))

    while remaining and len(selected_indices) < k:
        best_idx = -1
        best_score = -1e9
        for idx in remaining:
            rel = cosine_similarity(q_vec, cand_vecs[idx])
            redundancy = (
                max(cosine_similarity(cand_vecs[idx], cand_vecs[s]) for s in selected_indices)
                if selected_indices
                else 0.0
            )
            mmr_score = lambda_mult * rel - (1.0 - lambda_mult) * redundancy
            if mmr_score > best_score:
                best_score = mmr_score
                best_idx = idx
        selected_indices.append(best_idx)
        remaining.remove(best_idx)

    return [cand_docs[i] for i in selected_indices]


if __name__ == "__main__":
    vs = StarMindVectorStore()
    vs.add_documents([
        Document("一线城市出差住宿上限600元。", {"id": 1}),
        Document("一线城市出差住宿上限为600元每晚。", {"id": 2}),  # 与id=1高度重复
        Document("出差餐饮补助每天100元无需发票。", {"id": 3}),
    ])
    diversified = mmr_search(vs, "出差报销标准", k=2, lambda_mult=0.5)
    print("MMR 多样化召回:", [(d.metadata["id"], d.page_content) for d in diversified])
    assert {d.metadata["id"] for d in diversified} != {1, 2}


**答案解析与评分标准：**

MMR 是 LangChain `vectorstore.max_marginal_relevance_search()` 的核心算法，特别适合汇总型问题（如「出差报销都有哪些规定？」），能把住宿、餐补、交通多维度同时召回。


### 作业 3（架构进阶）：作业三：实现多租户命名空间隔离管理器（TenantVectorRegistry）

**题目要求：**
星辰智脑作为 SaaS 平台服务多家客户企业，要求每个租户拥有物理隔离的向量 Collection。实现 `TenantVectorRegistry(base_dir)`：提供 `get_store(tenant_id)` 方法，按 `tenant_id` 懒加载并缓存各自独立的 `StarMindVectorStore`，并校验 `tenant_id` 防止路径穿越攻击（如 `../etc`）。

**思路提示：** 用正则校验 `tenant_id` 仅允许字母/数字/下划线，存储目录设为 `base_dir / tenant_id`。


In [ ]:
"""Day 29 作业三参考答案：防路径穿越的多租户物理隔离向量库管理器"""
import re
import sys
import tempfile
from pathlib import Path
from typing import Dict

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day28_loaders import Document
from src.day29_vector_store import StarMindVectorStore


class TenantVectorRegistry:
    TENANT_RE = re.compile(r"^[a-zA-Z0-9_-]{2,32}$")

    def __init__(self, base_dir: str | Path) -> None:
        self.base_dir = Path(base_dir)
        self._cache: Dict[str, StarMindVectorStore] = {}

    def get_store(self, tenant_id: str) -> StarMindVectorStore:
        if not self.TENANT_RE.match(tenant_id):
            raise ValueError(f"非法 tenant_id: {tenant_id}")
        if tenant_id not in self._cache:
            tenant_dir = self.base_dir / tenant_id
            self._cache[tenant_id] = StarMindVectorStore(
                collection_name=f"kb_{tenant_id}",
                persist_directory=tenant_dir,
            )
        return self._cache[tenant_id]


if __name__ == "__main__":
    with tempfile.TemporaryDirectory() as tmpdir:
        reg = TenantVectorRegistry(tmpdir)
        reg.get_store("acme_corp").add_documents([Document("ACME机密报价单")])
        assert reg.get_store("beta_corp").count() == 0
        print("多租户物理隔离校验通过！")


**答案解析与评分标准：**

ToB SaaS 场景下，大客户通常要求物理级（独立 Collection / 独立 Partition）数据隔离而非仅靠 `where` 逻辑过滤，同时严防路径穿越漏洞。


---

## 🎯 老陈收尾复盘

今天大家把向量检索最核心的三大概念——余弦相似度、HNSW 分层近邻图、元数据预/后过滤（`where={'dept': '财务部'}`）全部亲手写代码跑通了。记住老陈的选型建议：本地原型和单机小库用 Chroma / Qdrant，千万级以上国产企业私有化上 Milvus 或 pgvector，纯算法离线批量计算用 FAISS。明天核心日，咱们把完整 RAG 系统端到端跑起来！
